# GT Raw Post-Processing

Goals:

1. Load each raw dataset, apply consistent preprocessing, and overlay manually selected `gt_raw` points for visualization.
2. Aggregate each `gt_raw` CSV by day (at most one GT point per day) and save a new CSV.
3. Generate an overlay plot with `all filtered voltage` + `daily-mean gt raw`.

In [23]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()),
    cwd,
 )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from master_arbeit_Di.ground_truth.c_gt_raw_process import (
    PreprocessConfig,
    process_many_gt_raw_files,
    process_many_gt_raw_regressions,
)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Z0057NPT\Documents\MA_code


In [24]:
PREPROCESS_CFG = PreprocessConfig(
    i_off=0.1,
    u_off=1.3,
    resample=1,
    data_filter_i_min=0.1,
    data_filter_U_min=1.4,
    data_filter_U_max=2.3,
    data_filter_T_min=50.0,
    data_filter_T_max=65.0,
    data_filter_h_since_last_start_min=0.5,
)

PREPROCESS_OUT = str(PROJECT_ROOT / "explore_data" / "output")
GT_RAW_DIR = PROJECT_ROOT / "master_arbeit_Di" / "ground_truth" / "output_backup" / "gt_raw"
OUTPUT_DIR = str(PROJECT_ROOT / "master_arbeit_Di" / "ground_truth" / "output_backup" / "gt_raw_processed")
SAVE_HTML = True

print("GT_RAW_DIR:", GT_RAW_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

GT_RAW_DIR: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw
OUTPUT_DIR: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed


## Configure Mapping: gt_raw CSV -> dataset

Specify the source dataset path for each `gt_raw` CSV file.

In [25]:
ITEMS = [
    {
        "dataset_path": str(PROJECT_ROOT / "explore_data" / "G6M2.parquet"),
        "gt_raw_csv": str(GT_RAW_DIR / "G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10.csv"),
    },
    {
        "dataset_path": str(PROJECT_ROOT / "explore_data" / "G6M2.parquet"),
        "gt_raw_csv": str(GT_RAW_DIR / "G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33.csv"),
    },
    {
        "dataset_path": str(PROJECT_ROOT / "explore_data" / "G6M2.parquet"),
        "gt_raw_csv": str(GT_RAW_DIR / "G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18.csv"),
    },
    {
        "dataset_path": str(PROJECT_ROOT / "explore_data" / "G1M1_new.parquet"),
        "gt_raw_csv": str(GT_RAW_DIR / "G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11.csv"),
    },
    {
        "dataset_path": str(PROJECT_ROOT / "explore_data" / "G1M1_new.parquet"),
        "gt_raw_csv": str(GT_RAW_DIR / "G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100.csv"),
    },
    {
        "dataset_path": str(PROJECT_ROOT / "explore_data" / "G1M1_new.parquet"),
        "gt_raw_csv": str(GT_RAW_DIR / "G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100.csv"),
    },
]

missing = [item["gt_raw_csv"] for item in ITEMS if not Path(item["gt_raw_csv"]).exists()]
print("Configured files:", len(ITEMS))
print("Missing gt_raw files:", len(missing))
for p in missing:
    print("  -", p)

Configured files: 6
Missing gt_raw files: 0


In [26]:
SUMMARY_DF = process_many_gt_raw_files(
    items=ITEMS,
    preprocess_output_dir=PREPROCESS_OUT,
    output_dir=OUTPUT_DIR,
    preprocess_config=PREPROCESS_CFG,
    save_html=SAVE_HTML,
)

print("Processed files:", len(SUMMARY_DF))

=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   C:\Users\Z0057NPT\Documents\MA_code\explore_data\output\G6M2_20260506_174732.parquet

=== GMpreprocess Pipeline Completed Successfully ===
[preprocess_once] 1096020 -> 359353 points.
=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   C:\Users\Z0057NPT\Documents\MA_code\explore_data\output\G6M2_20260506_174749.parquet

=== GMpreprocess Pipeline Completed Success

In [ ]:
SUMMARY_DF.style \
    .format({"raw_points": "{:.0f}", "daily_points": "{:.0f}"}) \
    .background_gradient(subset=["raw_points", "daily_points"], cmap="YlOrRd")

,dataset_path,dataset_name,gt_raw_csv,raw_points,daily_points,daily_csv,raw_overlay_html,daily_overlay_html
0,..\\..\\explore_data\\G6M2.parquet,G6M2,output_backup\\gt_raw\\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10.csv,30218,347,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_mean.csv,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__raw_overlay.html,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_overlay.html
1,..\\..\\explore_data\\G6M2.parquet,G6M2,output_backup\\gt_raw\\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33.csv,597,10,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_mean.csv,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__raw_overlay.html,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_overlay.html
2,..\\..\\explore_data\\G6M2.parquet,G6M2,output_backup\\gt_raw\\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18.csv,504,6,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_mean.csv,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__raw_overlay.html,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_overlay.html
3,..\\..\\explore_data\\G1M1_new.parquet,G1M1_new,output_backup\\gt_raw\\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11.csv,388019,1042,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_mean.csv,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__raw_overlay.html,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_overlay.html
4,..\\..\\explore_data\\G1M1_new.parquet,G1M1_new,output_backup\\gt_raw\\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100.csv,109407,333,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_mean.csv,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__raw_overlay.html,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_overlay.html
5,..\\..\\explore_data\\G1M1_new.parquet,G1M1_new,output_backup\\gt_raw\\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100.csv,4074,20,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_mean.csv,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__raw_overlay.html,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_overlay.html


In [ ]:
REG_SUMMARY_DF = process_many_gt_raw_regressions(
    items=ITEMS,
    preprocess_output_dir=PREPROCESS_OUT,
    output_dir=OUTPUT_DIR,
    preprocess_config=PREPROCESS_CFG,
    save_html=SAVE_HTML,
)

print("Regression processed files:", len(REG_SUMMARY_DF))

=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   explore_data\output\G6M2_20260506_172936.parquet

=== GMpreprocess Pipeline Completed Successfully ===
[preprocess_once] 1096020 -> 359353 points.
=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   explore_data\output\G6M2_20260506_172950.parquet

=== GMpreprocess Pipeline Completed Successfully ===
[preprocess_once] 1096020 -> 359353 points.
=== 1. Loading & P

In [14]:
REG_SUMMARY_DF.style \
    .format({
        "daily_points": "{:.0f}",
        "slope_v_per_day": "{:.6g}",
        "slope_uv_per_h": "{:.3f}",
        "intercept_v": "{:.4f}",
        "r2": "{:.4f}",
    }) \
    .background_gradient(subset=["slope_uv_per_h", "r2"], cmap="Blues")

,dataset_path,dataset_name,gt_raw_csv,daily_points,slope_v_per_day,slope_uv_per_h,intercept_v,r2,daily_regression_full_coverage_csv,daily_regression_overlay_html
0,..\\..\\explore_data\\G6M2.parquet,G6M2,output_backup\\gt_raw\\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10.csv,347,5.21645e-05,2.174,1.6060,0.4291,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_overlay_regression.html
1,..\\..\\explore_data\\G6M2.parquet,G6M2,output_backup\\gt_raw\\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33.csv,10,0.000135377,5.641,1.7369,0.9641,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_overlay_regression.html
2,..\\..\\explore_data\\G6M2.parquet,G6M2,output_backup\\gt_raw\\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18.csv,6,0.00018156,7.565,1.7983,0.9857,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv,\output_backup\gt_raw_processed\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_overlay_regression.html
3,..\\..\\explore_data\\G1M1_new.parquet,G1M1_new,output_backup\\gt_raw\\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11.csv,1042,2.66314e-05,1.110,1.6207,0.8364,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_overlay_regression.html
4,..\\..\\explore_data\\G1M1_new.parquet,G1M1_new,output_backup\\gt_raw\\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100.csv,333,0.000141105,5.879,1.7596,0.9215,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_overlay_regression.html
5,..\\..\\explore_data\\G1M1_new.parquet,G1M1_new,output_backup\\gt_raw\\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100.csv,20,0.000209184,8.716,1.9095,0.8733,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv,\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_overlay_regression.html
